# 逐章量化 demo：Colab GPU

运行时选择 GPU，再从上到下执行。此Notebook自带20章及新增主题的27个demo源码，无需另上传工程。采用PyTorch CUDA数值模拟，默认FP64用于数学核验；不测试原生INT4/FP4 Tensor Core加速。当前发布仅经CPU验证，GPU测试由本Notebook实际执行并记录。

## 1. 解包本次源码快照

这是生成单元，包含压缩源码。先核对包与文件SHA-256，再写入新建的临时项目目录；不会覆盖已有工程。

In [ ]:
import base64, hashlib, io, json, os, sys, tempfile, zipfile
from pathlib import Path

# 包内仅有公开教学源码与文档；校验失败立即停止。
PACKAGE_SHA256 = '1cdf787c6bf67f55082f6c6dcb76c0ac640642e21a7af912de1a7edf12c0408d'
FILE_SHA256 = {'constraints-cpu.txt': 'a539311fb4199ddf63f61a757c24a3c4efe714b3a9f16bb23692dc037d27fceb', 'docs/00_基础与全景.md': 'b880ac88bc227f421cac705f7ae812925046870ba021bfcbaf58727cea2e7fa3', 'docs/01_均匀量化与误差.md': '131c84b3b77e473997607094b67e84c229292a81de823f127f1a9aeccc3f949f', 'docs/02_GPTQ与二阶误差补偿.md': 'd2ac3b5e7ff4097ac689358dcb396b473c542790862d7325321ed8b63e44966d', 'docs/03_离群值与缩放.md': '92b20ef9d0adbb53f4d9e5e12cfe7c7062acf3f71d1015522ef2474d53edfad2', 'docs/04_可学习校准与HQQ.md': 'a3aa63a1cde6e95d54007ac53af29db98c8310e2820caccb595adb0e6bd7d066', 'docs/05_旋转与非相干化.md': 'bf52d3e458e16dde4df2f30b0be015635a010e3c9b465a522f9875ef5d63fa91', 'docs/06_码本_稀疏与混合精度.md': '7c1b0892f07eb223104689070701deb1e8fadafc9a0899d3ddab03a600688146', 'docs/07_QAT_QLoRA与原生低比特.md': '0c07919d8c41f46c75330e2046efe2fca8c6864b6cabdb147241b3aa669c8454', 'docs/08_KV缓存与Attention量化.md': '4287881e601a23d472aadbe39870b504d00949a9c4beee5d61274b48685f3889', 'docs/09_浮点格式与推理系统.md': 'ce0882d3abf600f29452845beb76c91ff8baa16a11bad75f7ec47346a2b3bf36', 'docs/10_实验_评估与选型.md': '56e7e170c06271ca3d1fbb22c9502211a4f91c470962a3dc416d70b7d37bc2cc', 'docs/11_练习与答案.md': '2509c4ddc65c1fd198a373bb085ad9245cb1ba5025ec6c5b41e2b03bd01ff863', 'docs/12_新进展_变换与编码.md': 'baeeb666e4f58cacc6ea998d0a5e9ab2ded1917804e854f268f750ec76d06b45', 'docs/13_新进展_FP4与误差修正.md': 'e15bb9bdfd3155d930f2cb98f0fba81ff9a843180311eeb22116067e3b724168', 'docs/14_新进展_训练与MoE.md': '232b85c2b552ae9b377afd2d4f45150ed5ccf2ae986fadbc734c7db76ac3cdc6', 'docs/15_新进展_KV与推理系统.md': '81660cff0e5354da930088284385e7be38c9e3292c2005f1e079963811fbb719', 'docs/16_未来趋势与研究路线.md': 'cb84f9143556df25d0c1c73b14abc62339b63565c17220ab933f48342a6256f2', 'docs/专题_LLM_int8_原理与实测.md': '52af89ac6573b736242566edc3c1bc76a857c00e4f7180149b73063cae798170', 'docs/科研进阶_高质量量化论文路线与清单.md': 'dea090d008690f0b384c812409d8d4a13b67ddf5318af0c556c2c5ad47fad79a', 'docs/附录A_推导所需的数学.md': '4e4cae69b5b1151e7350cd1bec7f60184b95b0d897c0edca5426ce7cdc016855', 'experiments/build_chapter_demos.py': '546d6c270ade38a3dc0aa1e84f16f4c9a271b8d77d3a15bf9f793498d20616ff', 'experiments/build_colab_chapter_demos.py': 'f8a6ba2aef5f53e6de9bc37354a27296cefca71f2f2557b39bc8f13fbfbc5190', 'experiments/build_technique_docs.py': 'a883e022d3d71473d6d26c4d49c697a874fa286e67649dacfad99fb936e52506', 'experiments/chapter_demo_catalog.py': '5f7266bcc35ea65cd88e8204efc96f02e6c4fb223f6deac8ae6eeec7cf0fa247', 'experiments/chapter_demo_lab.py': '5ba215f8484839772906f766804bf6a8316632d912a9853660e6ad8ceff75d11', 'experiments/check_project.py': '23ca5e058a579a378c328274cd47ab71d91d689b7abb189df6d49909e236d883', 'experiments/cuda_cases.py': '9a8f70939d1223624882672e2af04ba8c9912503a0665d7df6bb3e56975985d6', 'experiments/cuda_kernels.py': 'f84b65d24ef69e8537d343b6f75b139a10330d952dfdb1b451fb32f3d8e83f2b', 'experiments/cuda_lab.py': '74b17e2bb3493e8848009a1701cef43a802b5697c4de7de009490355110653a7', 'experiments/cuda_runtime.py': '2ce869c9c4ab62c1364c68a631efc75b203fa7b6dad8afea4167c2b2667c44b0', 'experiments/cuda_triton.py': '038b7b5572bff641786d6539c5794a0c435ac4544169c601a61bf0e4a4fcc166', 'experiments/deep_dive_lab.py': 'f9cd0243010310c37c24939748deba984b374876e14e47494e37ccf3cf85e96c', 'experiments/demo_runtime.py': '89e46156a48f6f54d80ec7f4e6ae402737af34e6c3270d6ee028d2f971e3da9a', 'experiments/demo_torch_kernels.py': 'ec360d3d9d75259c62615fa690f05d0864c456b4f270e2fa913c1dae92d11e10', 'experiments/demos/appendix_math.py': 'ed16f46b02e3c16b9537b33ef90f79cd40bcdcf8040472de99d5c459be398a19', 'experiments/demos/ch00.py': '580ea848bc90e6ab7033ba6f5c915ae88334fcd57173770bd7feed8364333c9e', 'experiments/demos/ch01.py': 'd8d72483782a047aed21270361ed95bb80b61ab9ff24a09e08cc0b1a5b45abc9', 'experiments/demos/ch02.py': 'f3058de94c32600649343dd0a6402083e54a3dbc80f35a6d81aa237b4d17dd2b', 'experiments/demos/ch03.py': '074077800f4021c473618d62f453ea4f507a34d653aeb97aab475011eec48ed2', 'experiments/demos/ch04.py': 'bd93f7f726c187f16c2c55d58508942d08892f030f0cf01fa56b7b9bda9893a3', 'experiments/demos/ch05.py': '35d67681b61f1814e9f1aee71099224c5e94600d343b095072720496052af7bf', 'experiments/demos/ch06.py': '722f2a63ef8b89607842abaa55467d72f8c6d0e9cf3e7eadfea807bf61c535fa', 'experiments/demos/ch07.py': 'f7747f36f40d47c2fb3cd80b8241eb25555464c076626879be4f1ee928548c20', 'experiments/demos/ch08.py': '48f59a004c6dd16bdefad560a5a98b9b7de06002354531dc95559dbc063ca7d8', 'experiments/demos/ch09.py': 'fcbfbd604f735eb10c8445f768e44e38bfc0abc5fc132610b581a026c3177401', 'experiments/demos/ch10.py': '11d9f303a75fe85c1f3b8984df99381c152395737b92e843262faa0c956a20b7', 'experiments/demos/ch11.py': 'f870dd5575ca166180a1ba8adaabfb7a9b534ee9fd400d04e6139f354f60382b', 'experiments/demos/ch12.py': 'fb8b0b3658db1031c9bb65185be68cb0bbdbe33c3266f58696ab02764ab3a2ae', 'experiments/demos/ch13.py': '50e21cc21c88a2bdff1762135dfd735bd3e16d830f32b9c805dcd24f78186d28', 'experiments/demos/ch14.py': 'a4be35f937e2040ddd731c1b4b19d62477a444ec250ea2f3a0151ecac9d8dd79', 'experiments/demos/ch15.py': '496e55f07763eee112024026431898faa454d6ac950290712152152a48fd5736', 'experiments/demos/ch16.py': '2109c05d1ddaa0296edeb3114401df3063f38eaf38b36747ba1f95c5ba473cfc', 'experiments/demos/four_over_six.py': '1277e292a36f4376a4b67bc0c74f413d0c2aaea2f89961c3b2191aad2c5919ed', 'experiments/demos/gpu_formats.py': '3ba13b12fe6b0df2443567775b8c4205de34619d5518339ddfecb4839a230173', 'experiments/demos/gradient_rounding.py': 'c0921881fc3ac311ae9bed1aad25bb1d7010adda4c920773553a8170e5cf8f0a', 'experiments/demos/int8.py': 'b6fc39ae5bdda0c75c4a13bbb030f86b87ab639f4f43c69b3e101ea279b13c22', 'experiments/demos/mr_gptq.py': '6579b55ef11bad6bd6782fb1907418096d2ed5476875a22742fa7b49e516d4b7', 'experiments/demos/oscillation.py': 'f37fa504b549dbdee82d3715f105348525a8cee22b187baa860e8349514aee69', 'experiments/demos/research.py': 'efb056afe0371cd93a8d22c00d6f828abb0574b03ca64320854a4883482e0fc1', 'experiments/demos/scale_layout.py': '5fb3a7197ae5dc19edb8ede406e7f5d161f10942b1d2c0482840993766989c49', 'experiments/demos/training_transpose.py': 'bfe98f5aa7f7eb7724a6392e07c1be6b17796312f8a9b0c8f86f2deb871f6159', 'experiments/inspect_run.py': 'dddc83e6e8b297fe3c7c2f4ddabd467f2c9ea5c8d2b60d0701b67a8d2160d7b9', 'experiments/latest_techniques.py': 'a577f31a6f1293d42365feae43a2a8f5a0681f0e6af2405441847c6595731c95', 'experiments/llm_int8_lab.py': 'bbc7b8d308b122a55193b1c30cbc154e8e08407de88e4b3241581ef0ada50ef4', 'experiments/native_nvfp4_probe.py': '565c3559539d62eaac2e4ef2188e54757ab66bd6f0b11d6b362b6970875983b3', 'experiments/outlook_scenarios.py': '6f5cb98ceec4e63be575793ec65f36dd439097a696aee47cf83153cb716c9fdb', 'experiments/quant_lab.py': 'fd19a8677f98d708e11a8a5600e046a21b4cadf4fea31ad546dbfa4a1d04b6bb', 'experiments/result_utils.py': '3cd1b298a3a539da99b4826eb61d3128a8ea95fb2c46d3bdb1a6c613fed0b2e4', 'experiments/technique_catalog.py': '112fdc0bc3add673ce0537f4ab7ff10d3ef573920ed5146d3283ee027b73668f', 'experiments/technique_kernels.py': '71e69b95ee6cf3318151226ba993b0e040ef7fd0a77e654d66317672b204cda3', 'experiments/technique_lab.py': '675ae2484e7f7fd1ad623144d07320c5da9222cc158e862cad22320ac303a90a', 'experiments/validation.py': '1ebceff9e90ec8ac3d4b66699e2617fbc09441d94a8275d635f380d6262bf982', 'requirements-demos.txt': 'dd3ea7b15eb35c23d7948fb9c4bd0dcca055a087ba788fb108ab0edd6f2dd99b', 'requirements.txt': 'bde68ffcfb53dd010e2ca394340aee83803b9f8e90ecfb32b1c600cd4d3db081', 'tests/test_demo_torch_kernels.py': 'af32b34cb01fdd89aaeebbb10922fef36321a8f5e9fd217398eef272c6d3f8f3'}
PACKAGE_B64 = '...'
package = base64.b64decode(PACKAGE_B64)
assert hashlib.sha256(package).hexdigest() == PACKAGE_SHA256
parent = '/content' if Path('/content').exists() else None
project = Path(tempfile.mkdtemp(prefix='quant_chapter_demos_', dir=parent))
with zipfile.ZipFile(io.BytesIO(package)) as archive:
    assert set(archive.namelist()) == set(FILE_SHA256)
    for name, expected in FILE_SHA256.items():
        destination = (project/name).resolve()
        if not destination.is_relative_to(project.resolve()):
            raise ValueError('非法归档路径')
        content = archive.read(name)
        assert hashlib.sha256(content).hexdigest() == expected, name
        destination.parent.mkdir(parents=True, exist_ok=True)
        destination.write_bytes(content)
os.chdir(project)
sys.path.insert(0, str(project))
print('源码目录：', project, '；核验文件数：', len(FILE_SHA256))

## 2. 检查环境与GPU

保留Colab预装的CUDA版PyTorch。若没有GPU，请先更改运行时；不要安装CPU专用torch构建。

In [ ]:
import importlib.util, subprocess
# 仅在缺包时安装，避免覆盖Colab已经可用的CUDA版本。
missing = [name for name in ('numpy', 'torch') if importlib.util.find_spec(name) is None]
if missing:
    subprocess.run([sys.executable, '-m', 'pip', 'install', *missing], check=True)
import torch
version = tuple(int(part) for part in torch.__version__.split('+')[0].split('.')[:2])
if version < (2, 2):
    raise RuntimeError('需要PyTorch 2.2+；请使用更新的Colab运行时。')
print('PyTorch:', torch.__version__, 'CUDA构建:', torch.version.cuda)
if not torch.cuda.is_available():
    raise RuntimeError('没有可用CUDA，请在Colab选择GPU运行时后重新运行。')
print('GPU:', torch.cuda.get_device_name(0))
probe = torch.ones((2, 2), device='cuda')
print('CUDA矩阵乘探针:', probe@probe)
torch.cuda.synchronize()

## 3. 选择章节并运行

`all`表示全部；可选00–16、math、int8、research，或gpu_formats、mr_gptq、four_over_six、training_transpose、gradient_rounding、oscillation、scale_layout。随机输入由CPU准备并搬到设备，数值计算用CUDA；打印和文件I/O仍在主机。

先运行默认示例。练习时在左侧文件浏览器打开解包目录下的 `experiments/demos/`，修改源码并保存，将下方 `PRACTICE=True` 后重新执行本单元，无需重建文档或重新解包。数值断言与设备检查仍会执行；如改变了题设，应相应推导新的预期结果。每次执行创建新结果目录，失败时也可直接跳到下载单元取回已有结果和错误日志。

In [ ]:
# 显式cuda失败不会改用cpu；练习模式仅放宽正文一致性检查。
CHAPTER = 'all'
DEVICE = 'cuda'
PRACTICE = False
RESULT_DIR = Path(tempfile.mkdtemp(prefix='run_', dir=project))
command = [sys.executable, '-B', 'experiments/chapter_demo_lab.py',
           '--chapter', CHAPTER, '--device', DEVICE,
           '--write-results', '--output-dir', str(RESULT_DIR)]
if PRACTICE:
    command.append('--practice')
# 日志先写磁盘，避免异常令已完成的输出消失；每轮有独立目录。
with (RESULT_DIR/'chapter_execution.log').open('w') as log:
    completed = subprocess.run(command, stdout=log, stderr=subprocess.STDOUT, text=True)
print((RESULT_DIR/'chapter_execution.log').read_text())
print('本轮结果目录：', RESULT_DIR)
completed.check_returncode()
report = json.loads((RESULT_DIR/'chapter_demo_results.json').read_text())
assert report['status'] == 'completed'
for row in report['results']:
    print('\n章节', row['id'], row['title'], row['runtime']['device'])
    print(row['output'])

## 4. GPU独立数值对照

在CUDA上比较PyTorch量化/缓存与原NumPy参考，并运行全部27个demo的设备核验。练习模式只跳过正文同步要求。失败时保留错误和测试日志，不把CPU通过视作GPU通过。

In [ ]:
environment = dict(os.environ, QUANT_DEMO_PRACTICE='1' if PRACTICE else '0')
with (RESULT_DIR/'gpu_tests.log').open('w') as log:
    checked = subprocess.run([sys.executable, '-m', 'unittest', 'discover', '-s', 'tests',
                              '-p', 'test_demo_torch_kernels.py', '-v'],
                             env=environment, stdout=log, stderr=subprocess.STDOUT, text=True)
print((RESULT_DIR/'gpu_tests.log').read_text())
checked.check_returncode()

## 5. 可选：原生NVFP4前后向探针

默认关闭。只有已配置TE 2.8的SM100/SM103环境才开启；脚本不会安装软件或回退到QDQ。T4/A100/H100无需运行此单元；SM120需匹配专用后端。此探针执行真实NVFP4 recipe，但不代替profiler指令确认或模型质量评测。

In [ ]:
RUN_NATIVE_NVFP4 = False
if RUN_NATIVE_NVFP4:
    with (RESULT_DIR/'native_execution.log').open('w') as log:
        native = subprocess.run([sys.executable, 'experiments/native_nvfp4_probe.py',
                                 '--output-dir', str(RESULT_DIR)],
                                stdout=log, stderr=subprocess.STDOUT, text=True)
    print((RESULT_DIR/'native_execution.log').read_text())
    native.check_returncode()
else:
    print('原生NVFP4探针未运行；上面的CUDA demo属于数值模拟。')

## 6. 下载结果

ZIP包含本轮逐章JSON、实际执行源码、测试日志和已运行的原生探针结果。JSON记录实际设备、运行状态及源码哈希；原生探针还记录实际张量dtype、种子和recipe。失败后也可运行本单元；`failed/partial/running`不能当作完整通过。

In [ ]:
import shutil
from google.colab import files
archive_path = shutil.make_archive(str(RESULT_DIR), 'zip', root_dir=RESULT_DIR)
files.download(archive_path)